In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
import joblib
from scipy.ndimage import convolve


In [ ]:
# Configuration
IMAGE_PATH = 'YOUR_IMAGE_PATH_HERE'  
OUTPUT_IMAGE_PATH = 'YOUR_OUTPUT_IMAGE_PATH_HERE'
SAVE_PATH = "YOUR_SAVE_PATH_HERE"
WIDTH = 4
LENGTH = 4
IMG_SIZE = 512  

# Load the pre-trained Random Forest Regressor model
MODEL_SAVE_PATH = 'YOUR_MODEL_PATH_HERE'
rf_model_loaded = joblib.load(MODEL_SAVE_PATH)

In [ ]:

def load_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    return img.astype(np.float32) / 255.0 if img is not None else None

def compute_mean(img):
    return np.mean(img)

image = tf.io.read_file(IMAGE_PATH)
image = tf.image.decode_png(image, channels=1)
image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
image = tf.cast(image, tf.float32) / 255.0
image = np.array(image)
image = np.squeeze(image)

means = []

for y_start in tqdm(range(0, image.shape[0] - LENGTH + 1, LENGTH), desc="Processing Image", unit="block"):
    for x_start in range(0, image.shape[1] - WIDTH + 1, WIDTH):
        block_image = image[y_start:y_start + LENGTH, x_start:x_start + WIDTH]
        mean_value = compute_mean(block_image)
        means.append(mean_value)

train_images = np.array(means).reshape(-1, 1)

In [ ]:
# Check if the model is fitted
if hasattr(rf_model_loaded, 'feature_importances_'):
    print("Model is fitted and ready to characterize.")
else:
    print("Model is not fitted. You need to fit it first.")

In [ ]:
if rf_model_loaded.n_features_in_ != 1:
    raise ValueError(f"Model expects {rf_model_loaded.n_features_in_} features, but received 1.")

In [ ]:

characterizeed_density = rf_model_loaded.characterize(train_images)
print(characterizeed_density)


In [ ]:
with open(SAVE_PATH, 'w') as file:
    density_idx = 0
    for y_start in range(0, image.shape[0] - LENGTH + 1, LENGTH):
        for x_start in range(0, image.shape[1] - WIDTH + 1, WIDTH):
            block = image[y_start:y_start + LENGTH, x_start:x_start + WIDTH]

            grayscale = np.mean(block) * 255

            density = characterizeed_density[density_idx]

            x_end = x_start + WIDTH
            y_end = y_start + LENGTH

            file.write(f"{x_start} {x_end} {y_start} {y_end} {grayscale:.2f} {density:.6f}\n")

            density_idx += 1

In [ ]:
density_map = np.zeros((image.shape[0], image.shape[1]))

density_idx = 0
for y_start in range(0, image.shape[0] - LENGTH + 1, LENGTH):
    for x_start in range(0, image.shape[1] - WIDTH + 1, WIDTH):
        density = characterizeed_density[density_idx]

        density_map[y_start:y_start + LENGTH, x_start:x_start + WIDTH] = density

        density_idx += 1

kernel = np.ones((3, 3)) / 9  
density_map = convolve(density_map, kernel)
image = convolve(image, kernel)

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
plt.imshow(image, cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(density_map, cmap='hot', interpolation='nearest')
cbar = plt.colorbar(ticks=[0, 20, 40, 60, 80], label="", fraction=0.05, pad=0.02)
cbar.ax.tick_params(labelsize=16)  
plt.title(" ")
plt.axis('off')  
plt.clim(-1, 81)  

plt.tight_layout()
plt.savefig(OUTPUT_IMAGE_PATH)
plt.show()